# 03 — Análise Exploratória e Preparação para PyTorch

Notebook atualizado para a versão final com **600 registros**, **3 classes** e vocabulário de **17.512 tokens**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import json
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

In [ ]:
csv_path = Path('../data/processed/amostra_rotulada.csv')
if not csv_path.exists():
    csv_path = Path('data/processed/amostra_rotulada.csv')

df = pd.read_csv(csv_path)
print('Registros:', len(df))
print(df['rotulo'].value_counts())
display(df.head())

## 1. Distribuição das classes

In [ ]:
contagem = df['rotulo'].value_counts().sort_index()
print(contagem)

contagem.plot(kind='bar')
plt.title('Distribuição de publicações por classe')
plt.xlabel('Classe')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

## 2. Tokenização

In [ ]:
STOPWORDS = {
    'de','da','do','das','dos','e','o','a','os','as','para','no','na','nos',
    'nas','que','em','um','uma','com','por','ao','aos','se','sua','seu',
    'pela','pelo','fica','art','nº','nr','ns','rs','ou','mais','mas','ser',
    'ter','foi','tem','são','este','esta','estes','estas','esse','essa',
    'esses','essas','todo','toda','todos','todas','não','também','já','sobre',
    'entre','até','muito','apenas','só','bem','ainda','quando','como','pelas',
    'pelos','num','numa','neste','nesta','deste','desta'
}

def tokenizar(texto):
    if not isinstance(texto, str):
        return []
    texto = texto.lower()
    texto = re.sub(r'[^a-zà-ú\s]', ' ', texto)
    tokens = texto.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

print(tokenizar('Decreto Municipal nº 4.521 - Fica decretado ponto facultativo.'))

## 3. Comprimento dos textos

In [ ]:
df['n_tokens'] = df['texto'].apply(lambda x: len(tokenizar(x)))
print(df['n_tokens'].describe())

df['n_tokens'].hist(bins=30)
plt.title('Distribuição do comprimento dos textos')
plt.xlabel('Número de tokens')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

## 4. Termos mais frequentes por classe

In [ ]:
for classe in sorted(df['rotulo'].unique()):
    contador = Counter()
    for texto in df[df['rotulo'] == classe]['texto']:
        contador.update(tokenizar(texto))
    print('
==', classe, '==')
    print(contador.most_common(15))

## 5. Vocabulário e label map

In [ ]:
classes = sorted(df['rotulo'].unique())
label2id = {c: i for i, c in enumerate(classes)}
df['rotulo_id'] = df['rotulo'].map(label2id)

treino_df, teste_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['rotulo_id']
)
print('Treino:', len(treino_df), '| Teste:', len(teste_df))
print(label2id)

In [ ]:
contador = Counter()
for texto in treino_df['texto']:
    contador.update(tokenizar(texto))

vocab = {'<PAD>': 0, '<UNK>': 1}
for token, freq in contador.most_common():
    if freq >= 2:
        vocab[token] = len(vocab)

print('Vocabulário:', len(vocab), 'tokens')

## 6. Codificação dos textos

In [ ]:
MAX_LEN = 60

def codificar_texto(texto, max_len=MAX_LEN):
    ids = [vocab.get(t, 1) for t in tokenizar(texto)]
    ids = ids[:max_len]
    ids += [0] * (max_len - len(ids))
    return ids

exemplo = codificar_texto(df.iloc[0]['texto'])
print(len(exemplo), exemplo[:20])

## 7. Conclusão

A base está balanceada em três classes e pronta para a Etapa 4. O vocabulário atual contém 17.512 tokens quando gerado pelo script `gerar_vocabulario.py` na base final.